## Tasks 3–4: Methods 3 (Historical) & 5 (EWMA Hisorical)

---
## From Other Noteboks

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

DATA_RETURNS = 'data/returns_clean.csv'
DATA_PRICES  = 'data/prices_clean.csv'

STOCKS   = ['ASML', 'SHELL', 'JPM']
INDEX    = ['STOXX50', 'SP500']
RISKY    = STOCKS + INDEX

SAMPLE_START = '2017-01-01'
SAMPLE_END   = '2026-03-31'

PORTFOLIO_VALUE = 1_000_000
TRADING_DAYS    = 252
LOAN_MOD_DUR    = 9.135099

WEIGHTS = {
    'ASML':    0.178207,
    'SHELL':   0.040000,
    'JPM':     0.320000,
    'STOXX50': 0.040000,
    'SP500':   0.221793,
    'LOAN':    0.200000,
}

ASSETS = list(WEIGHTS.keys())
w      = np.array([WEIGHTS[c] for c in ASSETS])
ALPHAS = [0.99, 0.975]

## Methods 3 (Historical Simulation)

In [2]:
returns = pd.read_csv(DATA_RETURNS, index_col=0, parse_dates=True)
ret     = returns.loc[SAMPLE_START:SAMPLE_END, ASSETS].dropna()
n       = len(ret)
# portfolio loss
loss     = -PORTFOLIO_VALUE * (ret.values @ w)
# per asset losses
loss_mat = -PORTFOLIO_VALUE * (ret.values * w)

In [3]:
results_ind  = {}
results_port = {}

for alpha in ALPHAS:
    K    = int(np.ceil(n * alpha))
    frac = K - n * alpha

    ls    = np.sort(loss)
    var_p = ls[K - 1]
    es_p  = (ls[K:].sum() + frac * var_p) / (n * (1 - alpha))
    results_port[alpha] = {'VaR': var_p, 'ES': es_p}

    results_ind[alpha] = {}
    for i, col in enumerate(ASSETS):
        ls_i  = np.sort(loss_mat[:, i])
        var_i = ls_i[K - 1]
        es_i  = (ls_i[K:].sum() + frac * var_i) / (n * (1 - alpha))
        results_ind[alpha][col] = {'VaR': var_i, 'ES': es_i}

In [6]:
print(f"{'Asset':<12}  {'VaR 99%':>10}  {'ES 99%':>10}  {'VaR 97.5%':>10}  {'ES 97.5%':>10}")
print('-' * 58)
for col in ASSETS:
    v99, e99   = results_ind[0.99][col]['VaR'],   results_ind[0.99][col]['ES']
    v975, e975 = results_ind[0.975][col]['VaR'],  results_ind[0.975][col]['ES']
    print(f"{col:<12}  {v99:>10,.0f}  {e99:>10,.0f}  {v975:>10,.0f}  {e975:>10,.0f}")
print('-' * 58)
v99, e99   = results_port[0.99]['VaR'],   results_port[0.99]['ES']
v975, e975 = results_port[0.975]['VaR'],  results_port[0.975]['ES']
print(f"{'PORTFOLIO':<12}  {v99:>10,.0f}  {e99:>10,.0f}  {v975:>10,.0f}  {e975:>10,.0f}")

Asset            VaR 99%      ES 99%   VaR 97.5%    ES 97.5%
----------------------------------------------------------
ASML              10,568      14,607       7,955      11,152
SHELL              2,084       3,197       1,434       2,306
JPM               15,058      22,768      10,911      16,918
STOXX50            1,464       1,961         904       1,472
SP500              7,600      11,185       5,342       8,263
LOAN                 941       1,263         648         973
----------------------------------------------------------
PORTFOLIO         28,182      43,288      20,860      31,813


## Methods 5 (Filtered Historical Simulation)

In [7]:
lam   = 0.94
r_arr = ret[ASSETS].values
sigma   = np.zeros((n, len(ASSETS)))

for i in range(len(ASSETS)):
    sigma[0, i] = np.var(r_arr[:, i])
    for t in range(1, n):
        sigma[t, i] = lam * sigma[t-1, i] + (1 - lam) * r_arr[t-1, i] ** 2

sigma = np.sqrt(sigma)

In [8]:
z        = r_arr / sigma
sigma_next = np.sqrt(lam * sigma[-1] ** 2 + (1 - lam) * r_arr[-1] ** 2)
r_sim    = z * sigma_next

In [9]:
loss_sim     = -PORTFOLIO_VALUE * (r_sim @ w)
loss_sim_ind = -PORTFOLIO_VALUE * (r_sim * w)

In [10]:
f_results_ind  = {}
f_results_port = {}

for alpha in ALPHAS:
    K    = int(np.ceil(n * alpha))
    frac = K - n * alpha

    ls    = np.sort(loss_sim)
    var_p = ls[K - 1]
    es_p  = (ls[K:].sum() + frac * var_p) / (n * (1 - alpha))
    f_results_port[alpha] = {'VaR': var_p, 'ES': es_p}

    f_results_ind[alpha] = {}
    for i, col in enumerate(ASSETS):
        ls_i  = np.sort(loss_sim_ind[:, i])
        var_i = ls_i[K - 1]
        es_i  = (ls_i[K:].sum() + frac * var_i) / (n * (1 - alpha))
        f_results_ind[alpha][col] = {'VaR': var_i, 'ES': es_i}

In [11]:
print(f"{'Asset':<12}  {'VaR 99%':>10}  {'ES 99%':>10}  {'VaR 97.5%':>10}  {'ES 97.5%':>10}")
print('-' * 58)
for col in ASSETS:
    v99, e99   = f_results_ind[0.99][col]['VaR'],   f_results_ind[0.99][col]['ES']
    v975, e975 = f_results_ind[0.975][col]['VaR'],  f_results_ind[0.975][col]['ES']
    print(f"{col:<12}  {v99:>10,.0f}  {e99:>10,.0f}  {v975:>10,.0f}  {e975:>10,.0f}")
print('-' * 58)
v99, e99   = f_results_port[0.99]['VaR'],   f_results_port[0.99]['ES']
v975, e975 = f_results_port[0.975]['VaR'],  f_results_port[0.975]['ES']
print(f"{'PORTFOLIO':<12}  {v99:>10,.0f}  {e99:>10,.0f}  {v975:>10,.0f}  {e975:>10,.0f}")

Asset            VaR 99%      ES 99%   VaR 97.5%    ES 97.5%
----------------------------------------------------------
ASML              13,091      18,193       9,462      13,971
SHELL              2,004       2,653       1,473       2,083
JPM               13,451      17,919      10,338      14,252
STOXX50            1,570       2,091       1,214       1,653
SP500              6,923       9,374       4,892       7,264
LOAN                 975       1,362         733       1,031
----------------------------------------------------------
PORTFOLIO         29,072      35,620      20,877      28,944
